# LeetCode #175: Combine Two Tables

https://leetcode.com/problems/combine-two-tables/

## Comparison of Approaches
| Approach | Time | Space |
|---|---|---|
| INNER JOIN | O(n×m) | O(result) |
| LEFT JOIN ★ | O(n×m) | O(result) |

## Understanding the Methods
### INNER JOIN
Would only return persons who have an address entry. Persons with no address would be excluded — incorrect per requirements.

### Optimal: LEFT JOIN ★
Use a LEFT JOIN from Person to Address on personId. This returns ALL persons, with NULL for city/state when no matching address exists. This is the canonical solution for "report all X regardless of whether Y exists."

## Solutions
### C#

In [ ]:
// SQL Problem — C# equivalent using LINQ
// Tables: Person(personId, lastName, firstName), Address(addressId, personId, city, state)

/*
-- SQL Solution:
SELECT p.firstName, p.lastName, a.city, a.state
FROM Person p
LEFT JOIN Address a ON p.personId = a.personId;
*/

// LINQ equivalent:
var result = from p in persons
             join a in addresses on p.personId equals a.personId into pa
             from addr in pa.DefaultIfEmpty()
             select new {
                 firstName = p.firstName,
                 lastName = p.lastName,
                 city = addr?.city,
                 state = addr?.state
             };
Console.WriteLine("LEFT JOIN returns all persons, NULL city/state when no address.");

### Python

In [ ]:
import pandas as pd

def combine_two_tables(person: pd.DataFrame, address: pd.DataFrame) -> pd.DataFrame:
    """
    SQL equivalent:
    SELECT firstName, lastName, city, state
    FROM Person LEFT JOIN Address ON Person.personId = Address.personId;
    """
    result = person.merge(address, on='personId', how='left')
    return result[['firstName', 'lastName', 'city', 'state']]

### Go

In [ ]:
package main

import "fmt"

type Person struct {
    PersonID  int
    LastName  string
    FirstName string
}

type Address struct {
    AddressID int
    PersonID  int
    City      string
    State     string
}

type Result struct {
    FirstName string
    LastName  string
    City      *string
    State     *string
}

// LEFT JOIN simulation in Go
func combineTwoTables(persons []Person, addresses []Address) []Result {
    addrMap := map[int]Address{}
    for _, a := range addresses {
        addrMap[a.PersonID] = a
    }
    var results []Result
    for _, p := range persons {
        r := Result{FirstName: p.FirstName, LastName: p.LastName}
        if a, ok := addrMap[p.PersonID]; ok {
            city, state := a.City, a.State
            r.City, r.State = &city, &state
        }
        results = append(results, r)
    }
    return results
}

func main() {
    fmt.Println("LEFT JOIN: all persons, nullable city/state")
}

### Rust

In [ ]:
use std::collections::HashMap;

#[derive(Debug)]
struct Person { person_id: i32, last_name: String, first_name: String }

#[derive(Debug)]
struct Address { person_id: i32, city: String, state: String }

#[derive(Debug)]
struct Result {
    first_name: String,
    last_name: String,
    city: Option<String>,
    state: Option<String>,
}

// LEFT JOIN simulation in Rust
fn combine_two_tables(persons: Vec<Person>, addresses: Vec<Address>) -> Vec<Result> {
    let addr_map: HashMap<i32, &Address> = addresses.iter().map(|a| (a.person_id, a)).collect();
    persons.iter().map(|p| {
        let addr = addr_map.get(&p.person_id);
        Result {
            first_name: p.first_name.clone(),
            last_name: p.last_name.clone(),
            city: addr.map(|a| a.city.clone()),
            state: addr.map(|a| a.state.clone()),
        }
    }).collect()
}

fn main() {
    println!("LEFT JOIN: returns all persons with nullable city/state");
}

## Examples

**Common:** Person has 2 rows; Address has 1 row matching person 1. Result: person 1 with city/state, person 2 with NULL/NULL.

**Slightly Complex:** Person has 3 rows; Address has 2 rows. Person with no address still appears with NULL city/state.

**Edge Time:** Person table empty → empty result set.

**Edge Space:** Address table empty → all persons returned with NULL city and NULL state.

**Almost-Impossible:** Multiple addresses per person (if allowed) — LEFT JOIN would duplicate person rows; use DISTINCT or subquery to deduplicate.